In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.edge.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException

In [20]:
def strip_parent(string):
    return string.split('(')[1]

def switch_date(driver, go_to_date):
    '''
    go_to_date[int]: 要切换到的日期
    '''
    date_xpath = '/html/body/div[1]/div/main/div/div[2]/div[1]/div[1]/div[1]/div[2]/div/div/div/div[2]/button[{num_date}]/div/span'.format(num_date=str(go_to_date))
    date_element = driver.find_element(By.XPATH, date_xpath)
    date = date_element.text
    date_element.click()
    print('已切换至',date,'日')
    
def get_match(driver, homescore_class_name, awayscore_class_name):
    '''
    获取完场比赛比分及盘口信息
    '''
    #打开新标签页获取盘口
    element = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/a/button'))) #Show More Button
    element.send_keys(Keys.CONTROL + Keys.RETURN)
    driver.switch_to.window(driver.window_handles[1])
    
    #获取比分及盘口
    home_score = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CLASS_NAME, homescore_class_name))).text
    away_score = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CLASS_NAME, awayscore_class_name))).text
    test_flags = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CLASS_NAME,'sc-eDWCr.itJafI')))
    handicap_H = driver.find_elements(By.CLASS_NAME,'sc-eDWCr.itJafI')[0].text #主盘口
    handicap_A = driver.find_elements(By.CLASS_NAME,'sc-eDWCr.itJafI')[1].text #客盘口
    value_home = driver.find_elements(By.CLASS_NAME,'sc-eDWCr.dsMMht')[2].text #主赔
    value_away = driver.find_elements(By.CLASS_NAME,'sc-eDWCr.dsMMht')[3].text #客赔    
    print(home_score, away_score, handicap_H ,handicap_A ,value_home ,value_away)
    
    #回到主页面
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    return [home_score, away_score, handicap_H, handicap_A, value_home, value_away]

def clean_handicap(handicap_value, value_home, value_away):
    '''
    清洗盘口信息
    '''
    if handicap_value == '0' or handicap_value == '-0':
        if value_home < value_away:
            handicap_value = '-0'
        elif value_away < value_home:
            handicap_value = '+0'
        else:
            handicap_value = '0'
    else:
        if not handicap_value.startswith('-'):
            handicap_value = '+'+handicap_value
    return handicap_value

def get_class_name(driver, flag):
    '''
    动态切换class_name
    '''
    CLASS_NAME = ''
    if flag == 'matches':
        try:
            driver.find_element(By.CLASS_NAME,'sc-hKwDye.LFQYO.sc-9199a964-1.bnnDyH') #比赛列表1
            CLASS_NAME = 'sc-hKwDye.LFQYO.sc-9199a964-1.bnnDyH'
        except NoSuchElementException:
            driver.find_element(By.CLASS_NAME,'sc-hLBbgP.dRtNhU.sc-9199a964-1.kusmLq') #比赛列表2
            CLASS_NAME = 'sc-hLBbgP.dRtNhU.sc-9199a964-1.kusmLq'
            
    elif flag == 'home_score':
        try:
            driver.find_element(By.CLASS_NAME,'sc-eDWCr.jgSsSj') #主比分1
            CLASS_NAME = 'sc-eDWCr.jgSsSj'
        except NoSuchElementException:
            driver.find_element(By.CLASS_NAME,'sc-eDWCr.iRYKkj') #主比分2
            CLASS_NAME = 'sc-eDWCr.iRYKkj'
            
    elif flag == 'away_score':
        try:
            driver.find_element(By.CLASS_NAME,'sc-eDWCr.hNUSos') #客比分1
            CLASS_NAME = 'sc-eDWCr.hNUSos'
        except NoSuchElementException:
            driver.find_element(By.CLASS_NAME,'sc-eDWCr.eRrOEk') #客比分2
            
    return CLASS_NAME

def init_class_name(driver):
    '''
    初始化class_name
    '''
    match_class_name = get_class_name(driver, 'matches')
    driver.find_element(By.CLASS_NAME, match_class_name).click()
    
    element = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/a/button'))) #Show More Button
    element.send_keys(Keys.CONTROL + Keys.RETURN)
    driver.switch_to.window(driver.window_handles[1])
    time.sleep(15)
    homescore_class_name = get_class_name(driver, 'home_score')
    awayscore_class_name = get_class_name(driver, 'away_score')
    
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    
    return match_class_name, homescore_class_name, awayscore_class_name

In [27]:
driver_path = r'D:\edgedriver_win64\msedgedriver.exe' #PC
#driver_path = r'D:\EdgeDriver\msedgedriver.exe' #company
driver = webdriver.Edge(service=Service(executable_path=driver_path))
driver.implicitly_wait(10)
driver.get("https://www.sofascore.com/")

#显示赔率
showodds = driver.find_element(By.CLASS_NAME,'slider')
showodds.click()
print('初始化完成！')

time.sleep(5)
switch_date(driver, 30) #切换日期
time.sleep(5)

#初始化class name
match_class_name, homescore_class_name, awayscore_class_name = init_class_name(driver)
print('获取CLASS_NAME成功')

初始化完成！
已切换至 30 日


NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":".sc-eDWCr.iRYKkj"}
  (Session info: MicrosoftEdge=106.0.1370.52)
Stacktrace:
Backtrace:
	Microsoft::Applications::Events::EventProperties::SetProperty [0x00007FF69DE3F0F2+15282]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DDDC7F2+1461634]
	Ordinal0 [0x00007FF69D9FC8A5+641189]
	Ordinal0 [0x00007FF69DA3B6B7+898743]
	Ordinal0 [0x00007FF69DA3BAAC+899756]
	Ordinal0 [0x00007FF69DA73DD7+1129943]
	Ordinal0 [0x00007FF69DA5891F+1018143]
	Ordinal0 [0x00007FF69DA711F9+1118713]
	Ordinal0 [0x00007FF69DA586F3+1017587]
	Ordinal0 [0x00007FF69DA2D104+839940]
	Ordinal0 [0x00007FF69DA2E76F+845679]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC98A18+135080]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC82878+44552]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC857BC+56652]


In [33]:
homescore_class_name

'sc-eDWCr.iRYKkj'

In [31]:
#模拟登录
profile = driver.find_element(By.XPATH,'/html/body/div[1]/div/header/div[1]/div/div[5]/div/a[3]')
profile.click()
time.sleep(5)

#Google登录
continue_with_google = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH, '/html/body/div[1]/div/main/div/div[2]/div/div/button[2]')))
continue_with_google.click()
for handle in driver.window_handles:
    driver.switch_to.window(handle)

#email
email = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.ID, "identifierId")))
email.send_keys('judd147@alumni.wfu.edu')
next_step = driver.find_element(By.XPATH,'/html/body/div[1]/div[1]/div[2]/div/div[2]/div/div/div[2]/div/div[2]/div/div[1]/div/div/button/span')
next_step.click()

#password
key = input('请输入密码:')
password = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.NAME, 'password')))
password.send_keys(key)
next_step = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div[1]/div[2]/div/div[2]/div/div/div[2]/div/div[2]/div/div[1]/div/div/button/span')))
next_step.click()

time.sleep(10)

#回到首页
driver.switch_to.window(driver.window_handles[0])
football = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div/header/div[2]/div/div/div[1]/ul[2]/li[1]/a')))
football.click()

expand = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[2]/div/div[2]/div')))
expand.click()
print('登录成功')

请输入密码:judd147t


TimeoutException: Message: 
Stacktrace:
Backtrace:
	Microsoft::Applications::Events::EventProperties::SetProperty [0x00007FF69DE3F0F2+15282]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DDDC7F2+1461634]
	Ordinal0 [0x00007FF69D9FC8A5+641189]
	Ordinal0 [0x00007FF69DA3B6B7+898743]
	Ordinal0 [0x00007FF69DA3BAAC+899756]
	Ordinal0 [0x00007FF69DA73DD7+1129943]
	Ordinal0 [0x00007FF69DA5891F+1018143]
	Ordinal0 [0x00007FF69DA711F9+1118713]
	Ordinal0 [0x00007FF69DA586F3+1017587]
	Ordinal0 [0x00007FF69DA2D104+839940]
	Ordinal0 [0x00007FF69DA2E76F+845679]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC98A18+135080]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC82878+44552]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC857BC+56652]
	Ordinal0 [0x00007FF69DAEA764+1615716]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DDE1D59+1483497]
	Microsoft::Applica

In [6]:
switch_date(driver, 30) #切换日期

已切换至 30 日


In [29]:
driver.close()
driver.switch_to.window(driver.window_handles[0])

In [32]:
#数据存储
#FIXME 1.如何根据数据库缺盘口的比赛寻找sofascore比赛
df_result = pd.DataFrame(columns=['比赛','盘口'])
first_scroll_flag = True
pinned_match_flag = True
amount_scrolled = 0
while pinned_match_flag:
    i = 0 #比赛index
    num_clicks = 0 #点击比赛次数
    driver.refresh() #刷新页面
    ActionChains(driver).scroll_by_amount(0, amount_scrolled).perform()
    time.sleep(5) #等待页面元素刷新
    
    matches = driver.find_elements(By.CLASS_NAME, match_class_name) #比赛列表
    container = driver.find_element(By.ID,'pinned-list-fade-target') #收藏夹
    item_list = container.text.split('\n') #收藏夹当前显示的比赛列表
    
    while num_clicks < 6:
        match = matches[i]
        i += 1
        match.click()
        num_clicks += 1
    
        info_list = get_match(driver, homescore_class_name, awayscore_class_name)
        home_score = info_list[0]
        away_score = info_list[1]
        handicap_H = info_list[2]
        handicap_A = info_list[3]
        value_home = info_list[4]
        value_away = info_list[5]

        #显示信息
        home_name = handicap_H.split(') ')[1]
        away_name = handicap_A.split(') ')[1]
        handicap_value = strip_parent(handicap_H.split(') ')[0])
        handicap_value = clean_handicap(handicap_value, value_home, value_away)
        print(home_name+' '+home_score+'-'+away_score+' '+away_name)
        print('盘口：', handicap_value)
        df_result = df_result.append({'比赛':home_name+' '+home_score+'-'+away_score+' '+away_name, '盘口':handicap_value}, ignore_index=True)
        
        #判断结束条件
        if home_name not in item_list:
            pinned_match_flag = False

    #划动比赛
    if first_scroll_flag:
        ActionChains(driver).scroll_by_amount(0, 700).perform()
        amount_scrolled += 700
        first_scroll_flag = False
    else:
        ActionChains(driver).scroll_by_amount(0, 350).perform()
        amount_scrolled += 350
    print('已划动')
    del matches

TimeoutException: Message: 
Stacktrace:
Backtrace:
	Microsoft::Applications::Events::EventProperties::SetProperty [0x00007FF69DE3F0F2+15282]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DDDC7F2+1461634]
	Ordinal0 [0x00007FF69D9FC8A5+641189]
	Ordinal0 [0x00007FF69DA3B6B7+898743]
	Ordinal0 [0x00007FF69DA3BAAC+899756]
	Ordinal0 [0x00007FF69DA73DD7+1129943]
	Ordinal0 [0x00007FF69DA5891F+1018143]
	Ordinal0 [0x00007FF69DA711F9+1118713]
	Ordinal0 [0x00007FF69DA586F3+1017587]
	Ordinal0 [0x00007FF69DA2D104+839940]
	Ordinal0 [0x00007FF69DA2E76F+845679]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC98A18+135080]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC82878+44552]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DC857BC+56652]
	Ordinal0 [0x00007FF69DAEA764+1615716]
	Microsoft::Applications::Events::EventProperty::EventProperty [0x00007FF69DDE1D59+1483497]
	Microsoft::Applica

In [ ]:
driver.quit()

In [98]:
df_result.drop_duplicates(subset=['比赛'], inplace=True)
df_result

,比赛,盘口
0,Leicester City 0-1 Manchester City,+1.25
1,AFC Bournemouth 2-3 Tottenham Hotspur,+0.75
2,Brentford 1-1 Wolverhampton,-0.25
3,Brighton & Hove Albion 4-1 Chelsea,+0.25
4,Crystal Palace 1-0 Southampton,-0.5
5,Newcastle United 4-0 Aston Villa,-0.75
6,RCD Mallorca 1-1 Espanyol,-0.25
7,UD Almería 3-1 Celta Vigo,+0.25
8,Cádiz CF 3-2 Atlético Madrid,+0.75
9,Werder Bremen 1-0 Hertha BSC,-0.25


In [ ]:
#球队字典
#FIXME UPDATE
def clean_teams(home, away, league_name):
    '''
    返回清洗后的球队名称
    '''
    teams_dict = {'日乙':{'群马温泉':'群马草津温泉','金泽塞维根':'金泽','琉球FC':'FC琉球'},
                  '美职联':{'辛辛那提FC':'辛辛那提','温哥华白帽':'温哥华白浪','堪萨斯城竞技':'堪萨斯城体育','波特兰伐木工':'波特兰伐木者'},
                  '日职联':{'鸟栖沙岩':'鸟栖砂岩','清水鼓动':'清水心跳','名古屋鲸八':'名古屋逆戟鲸'},
                  '阿甲':{'普拉腾斯':'普拉滕斯竞技','泰格雷':'老虎竞技','竞技俱乐部':'竞技','圣塔菲联':'圣菲联','巴拉卡斯中央队':'巴拉卡斯中央',
                               '科隆竞技':'哥伦布竞技','铁路工场':'塔列雷斯','阿尔多西维':'阿尔多希维','科尔多瓦中央SDE':'科尔多瓦中央',
                               '阿根廷独立':'独立','萨尔米安杜':'萨米恩托','飓风队':'飓风','帕特罗纳图':'天主教青年','防御与正义':'国防与司法'},
                  '德甲':{'莱比锡红牛':'RB莱比锡'},
                  '西甲':{'维戈塞尔塔':'塞尔塔','马洛卡':'马略卡','阿尔梅利亚':'阿尔梅里亚','巴利亚多利德':'巴拉多利德','加的斯':'加迪斯'},             
                  '英超':{'南安普敦':'南安普顿','曼彻斯特联':'曼联','曼彻斯特城':'曼城','莱切斯特城':'莱斯特城','托特纳姆热刺':'热刺'},
                  '法甲':{},
                  '意甲':{'克雷莫纳':'克雷莫内塞'},
                  '欧冠':{'里斯本竞技':'葡萄牙体育','托特纳姆热刺':'热刺','比尔森':'比尔森胜利','莱比锡红牛':'RB莱比锡',
                           '萨尔茨堡':'萨尔茨堡红牛','格拉斯哥流浪者':'流浪者','曼彻斯特城':'曼城'},
                  '欧联':{'谢里夫':'蒂拉斯波尔警长','曼彻斯特联':'曼联','博德闪耀':'博多格林特','PSV埃因霍温':'埃因霍温'},
                  '欧协联':{},#FIXME
                  '德乙':{'不伦瑞克':'布伦瑞克'},
                  '英冠':{'加的夫城':'卡迪夫城','布里斯托城':'布里斯托尔城','西布罗姆维奇':'西布朗'},
                  '西乙':{'格拉纳达GF':'格拉纳达','米兰迪斯':'米兰德斯','安道尔CF':'FC安道尔','阿尔巴切特':'阿尔瓦塞特','特內里费':'特内里费'},
                  '巴甲':{'奥瓦':'阿瓦伊','科里蒂巴':'库里蒂巴','布拉干蒂诺RB':'布拉甘蒂诺红牛','戈伊亚斯':'戈亚斯','福塔雷萨':'福塔莱萨','库亚巴':'奎尔巴'},
                  '墨超':{'老虎大学':'墨西哥老虎','马萨特兰FC':'马萨特兰','蒙特瑞':'蒙特雷','墨西哥美洲':'美洲','阿苏尔':'蓝十字',
                            '拿加沙':'内卡萨','圣路易斯竞技':'圣路易斯'},
                  '葡超':{'波尔蒂芒尼斯':'波尔蒂芒人','沙维什':'沙维斯','吉维森特':'吉尔维森特','里奥阿维':'阿维河','里斯本竞技':'葡萄牙体育',
                              '卡沙比亞':'卡萨皮亚','维薛拿':'维泽拉','费雷拉':'帕索斯费雷拉','马里迪莫':'马德拉航海'},
                  '荷甲':{'埃门':'埃蒙','福图纳锡塔德':'锡塔德幸运','PSV埃因霍温':'埃因霍温','维迪斯':'维特斯'},
                  '瑞典超':{'韦纳穆':'瓦纳默','IFK哥德堡':'哥德堡','AIK索尔纳':'索尔纳'},
                  '挪超':{'奥德':'奥特','博德闪耀':'博多格林特','格里姆斯塔':'谢夫','桑纳菲尤尔':'桑德菲杰','萨尔普斯堡':'萨普斯堡'},
                  '比甲':{'奥德赫维里':'奥哈瓦里','聚尔特瓦雷赫姆':'威尔郡','沙勒罗瓦':'沙勒鲁瓦','瑟兰联':'塞莱恩'},
                  '智甲':{'尤尼昂':'拉卡勒拉联','科金博':'科金博联合','维尼亚德马埃弗顿':'比尼亚德尔马埃弗顿','库里科':'库里科联合',
                           '塞雷那':'拉塞雷纳','奴伯伦斯':'纽布伦斯','希金斯':'奥希金斯','科布雷索':'科布雷萨尔','奥达斯':'奥达科斯意大利人',
                           '华奇巴托':'瓦奇巴托'},
                  '亚冠':{},#FIXME
                  '解放者杯':{},#FIXME
                  '南球杯':{'德尔瓦耶独立':'山谷独立'},#FIXME
                  '中超':{}
                  #Last Edit: 10/02/2022
                  }
    if league_name in teams_dict:
        for key, value in teams_dict[league_name].items():
            home = home.replace(key, value)
            away = away.replace(key, value)
    return home, away